In [1]:
import os
import json
import numpy as np
import pandas as pd

import shutil

import matplotlib.pyplot as plt

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    f1_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

In [2]:
SOURCE_FILE_NAME = "annotations.json"
FILTERED_FILE_NAME = "annotations_filtered.json"

ACD_EPOCHS_CSV = "acd_epochs.csv"
ACSA_EPOCHS_CSV = "acsa_epochs.csv"

# Diagram file paths
VIS_DATA_FILE_NAME = "token_lengths.json"
TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME = "token_length_distribution.png"

ACD_METRICS_PER_EPOCH = "acd_metrics_curves.png"
ACSA_METRICS_PER_EPOCH = "acsa_metrics_curves.png"

# Configurable Model Identifier: "classla/bcms-bertic" or "xlm-roberta-base"
MODEL_NAME = "classla/bcms-bertic"
SEED = 42
MAX_LEN = 512
EPOCHS = 10
SAVE_EPOCHS = 2
LR = 2e-5
WARMUP_STEPS = 0.1
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16


# Paths to save trained models
SAVE_DIR = "saved_models"
SAVE_DIR_ACD = f"./{SAVE_DIR}/acd"
SAVE_DIR_ACSA = f"./{SAVE_DIR}/acsa"

ALL_CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici",
    "Izgled", "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena"
]

POLARITIES_MAP = {
    "Neutralan": 0, "Pozitivan": 1, "Negativan": 2, "Konflikt": 3
}

INV_POLARITIES_MAP = {
    val: key for key, val in POLARITIES_MAP.items()
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(SOURCE_FILE_NAME, "r", encoding="utf-8") as f:
    data = json.load(f)

total_reviews = 0
over_512_count = 0
max_tokens_found = 0
all_lengths = []

print("Starting comment analysis...\n")

for item in data:
    text = item["comment"]
    review_status = item["review_status"]

    if review_status == "NE":
        continue

    total_reviews += 1

    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=False)
    num_tokens = len(token_ids)

    all_lengths.append(num_tokens)

    if num_tokens > 512:
        over_512_count += 1
        item["review_status"] = "NE"

    if num_tokens > max_tokens_found:
        max_tokens_found = num_tokens

percent = (over_512_count / total_reviews) * 100
mean_length = sum(all_lengths) / total_reviews

print("=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===")
print(f"Total number of reviews: {total_reviews}")
print(f"Number of reviews that go OVER 512 tokens: {over_512_count} ({percent:.2f}%)")
print(f"Mean review token length: {mean_length:.1f} tokena")
print(f"Longest reviews has: {max_tokens_found} tokens")

with open(FILTERED_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

viz_data = {
    "total_reviews": total_reviews,
    "over_512_count": over_512_count,
    "mean_length": mean_length,
    "max_length": max_tokens_found,
    "all_lengths": all_lengths
}

with open(VIS_DATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(viz_data, f, indent=4, ensure_ascii=False)

print(f"\nSuccessfully saved visualization data to '{VIS_DATA_FILE_NAME}'.")

plt.figure(figsize=(10, 6))
plt.hist(all_lengths, bins=40, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(x=512, color='red', linestyle='--', linewidth=2, label='Token limit (512)')

plt.title('Distribution of token length in reviews', fontsize=14)
plt.xlabel('No. tokens', fontsize=12)
plt.ylabel('No. reviews', fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig(TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME, dpi=300)
plt.close()

print(f"Successfully saved diagram to '{TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME}'.")

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1165 > 512). Running this sequence through the model will result in indexing errors


Starting comment analysis...

=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===
Total number of reviews: 6454
Number of reviews that go OVER 512 tokens: 11 (0.17%)
Mean review token length: 66.2 tokena
Longest reviews has: 1177 tokens

Successfully saved visualization data to 'token_lengths.json'.
Successfully saved diagram to 'token_length_distribution.png'.


In [ ]:
# ===========================================
# 1. DATA PREPARATION (Document-Level Splitting)
# ===========================================
def load_and_split_data(json_file_path, seed=SEED):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Filter unreviewed reviews
    valid_data = [item for item in data if item.get("review_status") != "NE"]

    # Split documents first to guarantee zero data leakage between splits
    train_items, temp_items = train_test_split(valid_data, test_size=0.20, random_state=seed)
    val_items, test_items = train_test_split(temp_items, test_size=0.50, random_state=seed)

    return train_items, val_items, test_items

def extract_absa_samples(items):
    acd_comments, acd_labels = [], []
    acsa_comments, acsa_categories, acsa_labels = [], [], []
    gold_tuples = []  # Set of (category, polarity_id) per document

    for item in items:
        comment = item["comment"]
        aspect_categories = item.get("aspect_categories", [])

        # ACD binary indicator vector
        acd_comments.append(comment)
        label_vector = [0.0] * len(ALL_CATEGORIES)
        doc_tuples = set()

        for aspect_category in aspect_categories:
            category = aspect_category.get("category")
            polarity = aspect_category.get("polarity")

            if category in ALL_CATEGORIES:
                idx = ALL_CATEGORIES.index(category)
                label_vector[idx] = 1.0

                # ACSA Comment + Category pair
                if polarity in POLARITIES_MAP:
                    pol_id = POLARITIES_MAP[polarity]
                    acsa_comments.append(comment)
                    acsa_categories.append(category)
                    acsa_labels.append(pol_id)
                    doc_tuples.add((category, pol_id))

        acd_labels.append(label_vector)
        gold_tuples.append(doc_tuples)

    return (acd_comments, acd_labels), (acsa_comments, acsa_categories, acsa_labels), gold_tuples

In [5]:
# ===========================================
# 2. LAZY DATASETS (Dynamic Padding)
# ===========================================
class LazyACDDataset(torch.utils.data.Dataset):
    def __init__(self, comments, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

class LazyACSADataset(torch.utils.data.Dataset):
    def __init__(self, comments, categories, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.categories = categories
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            text_pair=self.categories[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [6]:
# ===========================================
# 3. METRICS & THRESHOLD OPTIMIZATION
# ===========================================
def get_compute_metrics_acd(threshold=0.5):
    def compute_metrics_acd(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        predictions = (probs > threshold).astype(int)
        acc = accuracy_score(labels, predictions)
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
        micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
        weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
        return {
            "acd_accuracy": acc,
            "acd_macro_f1": macro_f1,
            "acd_macro_precision": macro_precision,
            "acd_macro_recall": macro_recall,
            "acd_micro_f1": micro_f1,
            "acd_weighted_f1": weighted_f1
        }
    return compute_metrics_acd

def compute_metrics_acsa(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    return {
        "acsa_accuracy": acc,
        "acsa_macro_f1": macro_f1,
        "acsa_macro_precision": macro_precision,
        "acsa_macro_recall": macro_recall,
        "acsa_micro_f1": micro_f1,
        "acsa_weighted_f1": weighted_f1
    }

def find_best_acd_thresholds_per_class(
    model, val_dataset, data_collator, device, categories
):
    model.to(device).eval()
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    best_thresholds = {}
    grid = np.arange(0.05, 0.90, 0.05)

    print("\n[Validation] Optimizing Per-Class ACD Thresholds:")
    for idx, cat in enumerate(categories):
        cat_probs = probs[:, idx]
        cat_labels = all_labels[:, idx]

        best_t = 0.5
        best_f1 = -1.0

        for thresh in grid:
            preds = (cat_probs > thresh).astype(int)
            f1 = f1_score(cat_labels, preds, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = thresh

        best_thresholds[cat] = float(best_t)
        print(
            f" - {cat:<15}: Threshold = {best_t:.2f} (Val F1: {best_f1:.4f})"
        )

    return best_thresholds

def evaluate_acd_overall(
    model, dataset, data_collator, thresholds, categories, device
):
    model.to(device).eval()
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    if isinstance(thresholds, dict):
        thresh_arr = np.array([thresholds[cat] for cat in categories])
    else:
        thresh_arr = np.array(thresholds)

    predictions = (probs > thresh_arr).astype(int)

    acc = accuracy_score(all_labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels, predictions, average="macro", zero_division=0
        )
    )
    micro_f1 = f1_score(
        all_labels, predictions, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        all_labels, predictions, average="weighted", zero_division=0
    )

    precisions, recalls, f1s, supports = precision_recall_fscore_support(
        all_labels, predictions, average=None, zero_division=0
    )

    category_metrics = []
    for idx, cat in enumerate(categories):
        category_metrics.append(
            {
                "category": cat,
                "threshold": float(thresh_arr[idx]),
                "f1": f1s[idx],
                "precision": precisions[idx],
                "recall": recalls[idx],
                "support": int(supports[idx]),
            }
        )

    category_metrics.sort(key=lambda x: x["f1"], reverse=True)

    return {
        "acd_accuracy": acc,
        "acd_macro_f1": macro_f1,
        "acd_macro_precision": macro_precision,
        "acd_macro_recall": macro_recall,
        "acd_micro_f1": micro_f1,
        "acd_weighted_f1": weighted_f1,
        "category_metrics": category_metrics
    }

In [7]:
# ===========================================
# 4. END-TO-END PIPELINE EVALUATION
# ===========================================
def evaluate_end_to_end(
    acd_model, acd_test_ds, acsa_model, tokenizer, test_comments, gold_tuples_list, acd_thresholds, batch_size=16
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    acd_model.to(device).eval()
    acsa_model.to(device).eval()

    if isinstance(acd_thresholds, dict):
        thresh_arr = np.array([acd_thresholds[cat] for cat in ALL_CATEGORIES])
    else:
        thresh_arr = np.array(acd_thresholds)

    # Run ACD inference in batches
    acd_loader = torch.utils.data.DataLoader(
        acd_test_ds, batch_size=batch_size, collate_fn=DataCollatorWithPadding(tokenizer)
    )
    all_acd_probs = []
    with torch.no_grad():
        for batch in acd_loader:
            batch.pop("labels", None)
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = acd_model(**inputs).logits
            all_acd_probs.append(torch.sigmoid(logits).cpu().numpy())
    all_acd_probs = np.vstack(all_acd_probs)

    # Collect positive pairs & track indexing
    acsa_inputs, pair_indices = [], []
    y_true, y_pred = [], []
    active_labels = [0, 1, 2, 3]
    pred_tuples_list = [set() for _ in range(len(test_comments))]

    for doc_idx, (comment, gold_tuples) in enumerate(zip(test_comments, gold_tuples_list)):
        gold_dict = dict(gold_tuples)
        acd_probs = all_acd_probs[doc_idx]

        for cat_idx, cat in enumerate(ALL_CATEGORIES):
            gold_pol = gold_dict.get(cat, -1)
            y_true.append(gold_pol)
            curr_flat_idx = len(y_true) - 1

            if acd_probs[cat_idx] > thresh_arr[cat_idx]:
                acsa_inputs.append((comment, cat))
                pair_indices.append(curr_flat_idx)
                y_pred.append(-1)
            else:
                y_pred.append(-1)

    # Run ACSA inference
    if acsa_inputs:
        acsa_preds = []
        for i in range(0, len(acsa_inputs), batch_size):
            batch_pairs = acsa_inputs[i:i + batch_size]
            encoded = tokenizer(
                [p[0] for p in batch_pairs],
                [p[1] for p in batch_pairs],
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                logits = acsa_model(**encoded).logits
                acsa_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())

        for idx, pred_pol in zip(pair_indices, acsa_preds):
            y_pred[idx] = pred_pol

    # Populate predicted tuple sets
    for flat_idx, pred_pol in enumerate(y_pred):
        if pred_pol in active_labels:
            doc_idx = flat_idx // len(ALL_CATEGORIES)
            cat = ALL_CATEGORIES[flat_idx % len(ALL_CATEGORIES)]
            pred_tuples_list[doc_idx].add((cat, pred_pol))

    # --- Existing Flat-Level Metrics ---
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="macro", zero_division=0
    )
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_true, y_pred, labels=active_labels, average="weighted", zero_division=0
    )

    # --- Tuple-Level Metrics ---
    total_tp, total_fp, total_fn = 0, 0, 0
    doc_f1s = []

    for gold_tuples, pred_tuples in zip(gold_tuples_list, pred_tuples_list):
        tp = len(gold_tuples & pred_tuples)
        fp = len(pred_tuples - gold_tuples)
        fn = len(gold_tuples - pred_tuples)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        doc_f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        doc_f1s.append(doc_f1)

    tuple_micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    tuple_micro_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    tuple_micro_f1 = (2 * tuple_micro_prec * tuple_micro_rec) / (tuple_micro_prec + tuple_micro_rec) if (tuple_micro_prec + tuple_micro_rec) > 0 else 0.0

    return {
        "e2e_macro_f1": macro_f1,
        "e2e_macro_precision": macro_precision,
        "e2e_macro_recall": macro_recall,
        "e2e_micro_f1": micro_f1,
        "e2e_micro_precision": micro_precision,
        "e2e_micro_recall": micro_recall,
        "e2e_weighted_f1": weighted_f1,
        # tuple metrics
        "tuple_macro_f1": float(np.mean(doc_f1s)),
        "tuple_micro_f1": tuple_micro_f1,
        "tuple_micro_precision": tuple_micro_prec,
        "tuple_micro_recall": tuple_micro_rec,
    }

In [8]:
# ===========================================
# 5. SAVE PER EPOCH TRAINING METRICS
# ===========================================
def save_epoch_metrics_to_csv(log_history, csv_filename, task_type="acd"):
    eval_logs = [entry for entry in log_history if "eval_loss" in entry]
    train_logs = [entry for entry in log_history if "loss" in entry]

    rows = []
    for eval_entry in eval_logs:
        epoch_num = int(round(eval_entry["epoch"]))

        # Pick the latest logged training step loss at or before this evaluation
        matching_train = [t for t in train_logs if t["epoch"] <= eval_entry["epoch"]]
        train_loss = matching_train[-1]["loss"] if matching_train else eval_entry.get("loss", np.nan)

        prefix = task_type.lower()
        prefix_cap = prefix.capitalize()

        row = {
            "Epoch": epoch_num,
            "Training Loss": train_loss,
            "Validation Loss": eval_entry.get("eval_loss", np.nan),
            f"{prefix_cap} Accuracy": eval_entry.get(f"eval_{prefix}_accuracy", np.nan),
            f"{prefix_cap} Macro F1": eval_entry.get(f"eval_{prefix}_macro_f1", np.nan),
            f"{prefix_cap} Macro Precision": eval_entry.get(f"eval_{prefix}_macro_precision", np.nan),
            f"{prefix_cap} Macro Recall": eval_entry.get(f"eval_{prefix}_macro_recall", np.nan),
            f"{prefix_cap} Micro F1": eval_entry.get(f"eval_{prefix}_micro_f1", np.nan),
            f"{prefix_cap} Weighted F1": eval_entry.get(f"eval_{prefix}_weighted_f1", np.nan),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    # Deduplicate evaluations for the same epoch, keeping the first (in-training) evaluation
    df = df.drop_duplicates(subset=["Epoch"], keep="first")

    df.to_csv(csv_filename, index=False, float_format="%.6f")
    print(f"Successfully generated and saved {csv_filename}")

In [9]:
# ===========================================
# 6. MAIN EXECUTION FLOW
# ===========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 1. Load and split data
train_items, val_items, test_items = load_and_split_data(FILTERED_FILE_NAME)

(acd_train_c, acd_train_l), (acsa_train_c, acsa_train_cat, acsa_train_l), _ = extract_absa_samples(train_items)
(acd_val_c, acd_val_l), (acsa_val_c, acsa_val_cat, acsa_val_l), _ = extract_absa_samples(val_items)
(acd_test_c, acd_test_l), (acsa_test_c, acsa_test_cat, acsa_test_l), gold_test_tuples = extract_absa_samples(test_items)

# 2. Build Datasets
acd_train_ds = LazyACDDataset(acd_train_c, acd_train_l, tokenizer)
acd_val_ds = LazyACDDataset(acd_val_c, acd_val_l, tokenizer)
acd_test_ds = LazyACDDataset(acd_test_c, acd_test_l, tokenizer)

acsa_train_ds = LazyACSADataset(acsa_train_c, acsa_train_cat, acsa_train_l, tokenizer)
acsa_val_ds = LazyACSADataset(acsa_val_c, acsa_val_cat, acsa_val_l, tokenizer)
acsa_test_ds = LazyACSADataset(acsa_test_c, acsa_test_cat, acsa_test_l, tokenizer)

In [10]:
print(len(acd_train_ds), len(acd_val_ds), len(acd_test_ds))

5154 644 645


In [11]:
# --- STAGE 1: TRAIN ACD ---
acd_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ALL_CATEGORIES),
    problem_type="multi_label_classification"
)
acd_args = TrainingArguments(
    output_dir="./results_acd",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acd_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acd_trainer = Trainer(
    model=acd_model,
    args=acd_args,
    train_dataset=acd_train_ds,
    eval_dataset=acd_val_ds,
    compute_metrics=get_compute_metrics_acd(0.5),
    data_collator=data_collator,
    processing_class=tokenizer
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

In [12]:
print("--- Starting Training for Stage 1: ACD ---")
acd_trainer.train()

--- Starting Training for Stage 1: ACD ---


Epoch,Training Loss,Validation Loss,Acd Accuracy,Acd Macro F1,Acd Macro Precision,Acd Macro Recall,Acd Micro F1,Acd Weighted F1
1,0.430362,0.399099,0.055901,0.163126,0.259645,0.138439,0.370917,0.283386
2,0.329813,0.327486,0.256211,0.322600,0.569321,0.273174,0.560817,0.474997
3,0.275395,0.287731,0.304348,0.466211,0.626666,0.415597,0.642857,0.587217
4,0.232731,0.252337,0.369565,0.529270,0.690495,0.472806,0.694763,0.659281
5,0.200434,0.237235,0.411491,0.597332,0.691010,0.561648,0.745737,0.720759
6,0.175740,0.218625,0.445652,0.644429,0.710022,0.614958,0.779230,0.762527
7,0.155112,0.213255,0.433230,0.643345,0.716866,0.608013,0.772114,0.755965
8,0.147758,0.210443,0.476708,0.678152,0.724108,0.658391,0.794461,0.783796
9,0.134389,0.207432,0.475155,0.675638,0.728683,0.644436,0.791388,0.780383
10,0.123247,0.205085,0.486025,0.683480,0.733279,0.653021,0.797924,0.787455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3230, training_loss=0.23985389017099198, metrics={'train_runtime': 779.8088, 'train_samples_per_second': 66.093, 'train_steps_per_second': 4.142, 'total_flos': 5456819481652920.0, 'train_loss': 0.23985389017099198, 'epoch': 10.0})

In [13]:
#log_df = pd.DataFrame(acd_trainer.state.log_history)
#print(log_df.head())

# Export Stage 1 metrics to CSV
save_epoch_metrics_to_csv(
    acd_trainer.state.log_history,
    ACD_EPOCHS_CSV,
    task_type="acd")

Successfully generated and saved acd_epochs.csv


In [14]:
# Find optimal per-class probability thresholds on validation set
best_acd_thresholds = find_best_acd_thresholds_per_class(
    acd_model, acd_val_ds, data_collator, device, ALL_CATEGORIES
)


[Validation] Optimizing Per-Class ACD Thresholds:
 - Baterija       : Threshold = 0.35 (Val F1: 0.9471)
 - Kamera         : Threshold = 0.45 (Val F1: 0.9034)
 - Ekran          : Threshold = 0.35 (Val F1: 0.7024)
 - Memorija       : Threshold = 0.10 (Val F1: 0.1455)
 - Zvučnici       : Threshold = 0.25 (Val F1: 0.7227)
 - Izgled         : Threshold = 0.20 (Val F1: 0.6452)
 - Hardver        : Threshold = 0.40 (Val F1: 0.7535)
 - Softver        : Threshold = 0.35 (Val F1: 0.8045)
 - Performanse    : Threshold = 0.45 (Val F1: 0.7122)
 - Cena           : Threshold = 0.20 (Val F1: 0.7978)
 - Opšta ocena    : Threshold = 0.55 (Val F1: 0.8774)


In [15]:
# Overall and per-category ACD evaluation on Test Set using per-class thresholds
acd_results = evaluate_acd_overall(
    acd_model,
    acd_test_ds,
    data_collator,
    best_acd_thresholds,
    ALL_CATEGORIES,
    device,
)
print("\n--- Standalone Overall Evaluation for ACD (Test Set) ---")
for metric, val in acd_results.items():
    if metric != "category_metrics":
        print(f"  {metric:<20}: {val:.4f}")

print("\n--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---")
print(
    f"{'Category':<15} | {'Thresh':<6} | {'F1 Score':<10} | {'Precision':<10} |"
    " {'Recall':<10} | {'Support':<8}"
)
print("-" * 75)
for item in acd_results["category_metrics"]:
    print(
        f"{item['category']:<15} | {item['threshold']:<6.2f} |"
        f" {item['f1']:<10.4f} | {item['precision']:<10.4f} |"
        f" {item['recall']:<10.4f} | {item['support']:<8}"
    )


--- Standalone Overall Evaluation for ACD (Test Set) ---
  acd_accuracy        : 0.4651
  acd_macro_f1        : 0.7217
  acd_macro_precision : 0.6759
  acd_macro_recall    : 0.7964
  acd_micro_f1        : 0.7940
  acd_weighted_f1     : 0.8088

--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---
Category        | Thresh | F1 Score   | Precision  | {'Recall':<10} | {'Support':<8}
---------------------------------------------------------------------------
Baterija        | 0.35   | 0.9469     | 0.9264     | 0.9683     | 221     
Kamera          | 0.45   | 0.9194     | 0.9006     | 0.9390     | 164     
Opšta ocena     | 0.55   | 0.8746     | 0.8558     | 0.8942     | 312     
Cena            | 0.20   | 0.7692     | 0.6838     | 0.8791     | 91      
Softver         | 0.35   | 0.7676     | 0.7282     | 0.8114     | 175     
Zvučnici        | 0.25   | 0.7395     | 0.6875     | 0.8000     | 55      
Hardver         | 0.40   | 0.7372     | 0.7014     | 0.7769     | 130     
Ekran   

In [16]:
# --- SAVE STAGE 1 (ACD MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 1 (ACD) model to {SAVE_DIR_ACD}...")
acd_trainer.save_model(SAVE_DIR_ACD)
tokenizer.save_pretrained(SAVE_DIR_ACD)

acd_config = {
    "best_thresholds": best_acd_thresholds,
    "categories": ALL_CATEGORIES,
}
with open(os.path.join(SAVE_DIR_ACD, "acd_config.json"), "w", encoding="utf-8") as f:
    json.dump(acd_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 1 (ACD) model to ./saved_models/acd...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
# --- STAGE 2: TRAIN ACSA ---
acsa_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(POLARITIES_MAP)
)
acsa_args = TrainingArguments(
    output_dir="./results_acsa",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acsa_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acsa_trainer = Trainer(
    model=acsa_model,
    args=acsa_args,
    train_dataset=acsa_train_ds,
    eval_dataset=acsa_val_ds,
    compute_metrics=compute_metrics_acsa,
    data_collator=data_collator,
    processing_class=tokenizer
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
print("\n--- Starting Training for Stage 2: ACSA ---")
acsa_trainer.train()


--- Starting Training for Stage 2: ACSA ---


Epoch,Training Loss,Validation Loss,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
1,0.641635,0.529491,0.827338,0.424995,0.417747,0.435552,0.827338,0.801433
2,0.453939,0.429370,0.863309,0.444372,0.430602,0.459892,0.863309,0.837321
3,0.327667,0.455210,0.879137,0.469831,0.607313,0.474160,0.879137,0.854594
4,0.330454,0.441449,0.867626,0.537450,0.639394,0.550946,0.867626,0.865360
5,0.189589,0.529545,0.876259,0.609132,0.634480,0.623697,0.876259,0.876447
6,0.140677,0.596232,0.883453,0.589696,0.631809,0.580738,0.883453,0.876966
7,0.152495,0.670736,0.874820,0.627566,0.622687,0.634059,0.874820,0.875801
8,0.101095,0.747791,0.869065,0.622347,0.610921,0.636658,0.869065,0.872156
9,0.050238,0.743626,0.872662,0.640504,0.637900,0.646569,0.872662,0.874994
10,0.028784,0.748660,0.876259,0.644655,0.637121,0.653235,0.876259,0.877638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7200, training_loss=0.25905667086442313, metrics={'train_runtime': 1829.8249, 'train_samples_per_second': 62.908, 'train_steps_per_second': 3.935, 'total_flos': 1.6011380454997824e+16, 'train_loss': 0.25905667086442313, 'epoch': 10.0})

In [19]:
# Export Stage 2 metrics to CSV
save_epoch_metrics_to_csv(
    acsa_trainer.state.log_history,
    ACSA_EPOCHS_CSV,
    task_type="acsa")

Successfully generated and saved acsa_epochs.csv


In [20]:
# Standalone ACSA (Oracle) Test Evaluation
print("\n--- Standalone Oracle Evaluation for ACSA (Test Set) ---")
acsa_test_results = acsa_trainer.evaluate(eval_dataset=acsa_test_ds)
print(acsa_test_results)


--- Standalone Oracle Evaluation for ACSA (Test Set) ---


Training Loss,Validation Loss,Epoch,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
0.028784,0.789751,10,0.869007,0.625343,0.619703,0.636848,0.869007,0.870246


{'eval_loss': 0.7897510528564453, 'eval_acsa_accuracy': 0.8690074274139096, 'eval_acsa_macro_f1': 0.6253428403763679, 'eval_acsa_macro_precision': 0.6197030123714228, 'eval_acsa_macro_recall': 0.6368475424097434, 'eval_acsa_micro_f1': 0.8690074274139096, 'eval_acsa_weighted_f1': 0.8702456135486878}


In [21]:
# --- SAVE STAGE 2 (ACSA MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 2 (ACSA) model to {SAVE_DIR_ACSA}...")
acsa_trainer.save_model(SAVE_DIR_ACSA)
tokenizer.save_pretrained(SAVE_DIR_ACSA)

acsa_config = {
    "polarities_map": POLARITIES_MAP
}
with open(os.path.join(SAVE_DIR_ACSA, "acsa_config.json"), "w", encoding="utf-8") as f:
    json.dump(acsa_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 2 (ACSA) model to ./saved_models/acsa...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
# --- END-TO-END PIPELINE EVALUATION ---
print("\n--- End-to-End Pipeline Evaluation (Test Set) ---")
e2e_results = evaluate_end_to_end(
    acd_model=acd_model,
    acd_test_ds=acd_test_ds,
    acsa_model=acsa_model,
    tokenizer=tokenizer,
    test_comments=acd_test_c,
    gold_tuples_list=gold_test_tuples,
    acd_thresholds=best_acd_thresholds,
)
print(e2e_results)


--- End-to-End Pipeline Evaluation (Test Set) ---
{'e2e_macro_f1': 0.5137569372992398, 'e2e_macro_precision': 0.4728425491401079, 'e2e_macro_recall': 0.5651787008844424, 'e2e_micro_f1': 0.6939668646452016, 'e2e_micro_precision': 0.6461001164144354, 'e2e_micro_recall': 0.74949358541526, 'e2e_weighted_f1': 0.695217027679544, 'tuple_macro_f1': 0.7097232104460847, 'tuple_micro_f1': 0.6939668646452016, 'tuple_micro_precision': 0.6461001164144354, 'tuple_micro_recall': 0.74949358541526}


In [23]:
# Creates a f'{SAVE_DIR}.zip' file in your current directory
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)

'/content/saved_models.zip'

In [24]:
df_acd = pd.read_csv(ACD_EPOCHS_CSV)
df_acsa = pd.read_csv(ACSA_EPOCHS_CSV)

# Setup plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
epochs = range(1, 11)

# ---------------------------------------------------------
# DIAGRAM: STAGE 1 (ACD)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACD Loss
ax1.plot(epochs, df_acd['Training Loss'], marker='o', color='#1f77b4', linewidth=2, label='Train Loss')
ax1.plot(epochs, df_acd['Validation Loss'], marker='s', color='#d62728', linewidth=2, linestyle='--', label='Val Loss')
ax1.set_title('Stage 1 (ACD): Training & Validation Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACD Metrics
ax2.plot(epochs, df_acd['Acd Macro F1'], marker='o', color='#2ca02c', linewidth=2, label='Macro F1')
ax2.plot(epochs, df_acd['Acd Micro F1'], marker='^', color='#ff7f0e', linewidth=2, label='Micro F1')
ax2.plot(epochs, df_acd['Acd Accuracy'], marker='d', color='#9467bd', linewidth=2, linestyle=':', label='Accuracy')
ax2.set_title('Stage 1 (ACD): Performance Metrics', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACD_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACD_METRICS_PER_EPOCH}")

# ---------------------------------------------------------
# DIAGRAM: STAGE 2 (ACSA)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACSA Loss
ax1.plot(epochs, df_acsa['Training Loss'], marker='o', color='#1f77b4', linewidth=2, label='Train Loss')
ax1.plot(epochs, df_acsa['Validation Loss'], marker='s', color='#d62728', linewidth=2, linestyle='--', label='Val Loss')
ax1.set_title('Stage 2 (ACSA): Training & Validation Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACSA Metrics
ax2.plot(epochs, df_acsa['Acsa Macro F1'], marker='o', color='#2ca02c', linewidth=2, label='Macro F1')
ax2.plot(epochs, df_acsa['Acsa Micro F1'], marker='^', color='#ff7f0e', linewidth=2, label='Micro F1')
ax2.plot(epochs, df_acsa['Acsa Accuracy'], marker='d', color='#9467bd', linewidth=2, linestyle=':', label='Accuracy')
ax2.set_title('Stage 2 (ACSA): Performance Metrics', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACSA_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACSA_METRICS_PER_EPOCH}")

Successfully generated and saved acd_metrics_curves.png
Successfully generated and saved acsa_metrics_curves.png
